# 🏦 Bank Customer Churn Prediction

**Run this top to bottom in Google Colab.** It performs the full analysis, trains the model, and
downloads a `churn_artifacts.zip` containing everything your GitHub repo needs.

| Step | What happens |
|---|---|
| 1 | Setup and version check |
| 2 | Load the dataset |
| 3 | Cleaning + feature engineering |
| 4 | Exploratory analysis (Seaborn) |
| 5 | Train and compare 3 models |
| 6 | Evaluate the Random Forest |
| 7 | Export artifacts for the repo, app and Power BI |

---
## 1. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.width", 120)

PALETTE = {0: "#4C9F70", 1: "#D1495B"}
RANDOM_STATE = 42

# Output folders (mirror the repo layout)
os.makedirs("reports/figures", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("powerbi", exist_ok=True)

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("seaborn     ", sns.__version__)
print("joblib      ", joblib.__version__)
print("\n^ Note these down. Step 7 writes a matching requirements.txt automatically.")

---
## 2. Load the dataset

Pick **one** of the three options below.

**Option A — upload manually (simplest).** Download `Churn_Modelling.csv` from
[Kaggle](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling), then run the next cell
and select the file.

In [ ]:
# OPTION A — manual upload
from google.colab import files
uploaded = files.upload()          # choose Churn_Modelling.csv
df = pd.read_csv("Churn_Modelling.csv")
print(df.shape)

**Option B — Kaggle API (no clicking, survives a runtime restart).**
Get `kaggle.json` from your Kaggle account page → Settings → Create New Token.

In [ ]:
# OPTION B — Kaggle API  (uncomment to use)
# from google.colab import files
# files.upload()                     # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d shrutimechlearn/churn-modelling -q
# !unzip -oq churn-modelling.zip
# df = pd.read_csv("Churn_Modelling.csv")
# print(df.shape)

**Option C — Google Drive**, if you've already saved the CSV there.

In [ ]:
# OPTION C — Google Drive  (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Churn_Modelling.csv')
# print(df.shape)

### First look at the data

In [ ]:
display(df.head())
print("\nShape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(f"Churn rate: {df['Exited'].mean():.2%}")
df.info()

In [ ]:
df.describe().T.round(2)

---
## 3. Cleaning and feature engineering

`RowNumber`, `CustomerId` and `Surname` are pure identifiers — they carry no signal and a tree model
will happily overfit to them, so they go.

Three engineered features are added:
- **BalancePerProduct** — how much money sits behind each product the customer holds
- **ZeroBalance** — a flag, because ~36% of customers hold exactly €0 and behave differently
- **TenurePerAge** — relationship length relative to life stage

In [ ]:
raw = df.copy()                                  # keep the original for the Power BI export

work = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
work["BalancePerProduct"] = work["Balance"] / work["NumOfProducts"].replace(0, 1)
work["ZeroBalance"] = (work["Balance"] == 0).astype(int)
work["TenurePerAge"] = work["Tenure"] / work["Age"]

print("Columns after cleaning:", work.shape[1])
work.head()

---
## 4. Exploratory analysis

Every chart is saved to `reports/figures/` and ends up in the zip at the end, ready to embed in
your README.

In [ ]:
def save(fig, name):
    fig.savefig(f"reports/figures/{name}.png", dpi=150, bbox_inches="tight")

# --- Churn balance ---
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=work, x="Exited", hue="Exited", palette=PALETTE, legend=False, ax=ax)
for p in ax.patches:
    ax.annotate(f"{p.get_height()/len(work):.1%}",
                (p.get_x() + p.get_width()/2, p.get_height()), ha="center", va="bottom")
ax.set_title("Churn distribution"); ax.set_xlabel("")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Stayed", "Churned"])
save(fig, "01_churn_distribution"); plt.show()

In [ ]:
# --- Churn rate by segment ---
cats = ["Geography", "Gender", "NumOfProducts", "IsActiveMember", "HasCrCard", "Tenure"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, cats):
    rate = work.groupby(col)["Exited"].mean().reset_index()
    sns.barplot(data=rate, x=col, y="Exited", hue=col, palette="Set2", legend=False, ax=ax)
    ax.set_title(f"Churn rate by {col}"); ax.set_ylabel("Churn rate")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.suptitle("Where does churn concentrate?", fontsize=15, y=1.01)
fig.tight_layout()
save(fig, "02_churn_by_category"); plt.show()

In [ ]:
# --- Numeric distributions split by churn ---
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flat, ["Age", "CreditScore", "Balance", "EstimatedSalary"]):
    sns.kdeplot(data=work, x=col, hue="Exited", fill=True, alpha=.35,
                palette=PALETTE, common_norm=False, ax=ax)
    ax.set_title(f"{col} by churn status")
fig.tight_layout()
save(fig, "03_numeric_distributions"); plt.show()

In [ ]:
# --- Age vs geography ---
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=work, x="Geography", y="Age", hue="Exited", palette=PALETTE, ax=ax)
ax.set_title("Age vs geography, split by churn")
ax.legend(title="Churned", labels=["No", "Yes"])
save(fig, "04_age_geography_box"); plt.show()

In [ ]:
# --- Correlation matrix ---
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(work.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8}, ax=ax)
ax.set_title("Correlation matrix")
save(fig, "05_correlation_heatmap"); plt.show()

### 📝 Write down what you actually see

Before moving on, fill these in from the charts above — these become the **Key Findings** section of
your README, and they're what an interviewer will ask about.

- Churn rate by geography: France ___%, Germany ___%, Spain ___%
- Churn rate: active members ___% vs inactive ___%
- Churn rate by product count: 1 → ___%, 2 → ___%, 3 → ___%, 4 → ___%
- The age band with the highest churn: ___


In [ ]:
# Quick numbers for the bullets above
for col in ["Geography", "IsActiveMember", "NumOfProducts", "Gender"]:
    print(f"\n--- Churn rate by {col} ---")
    print((work.groupby(col)["Exited"].agg(["mean", "count"])
             .rename(columns={"mean": "churn_rate", "count": "customers"})
             .assign(churn_rate=lambda d: (d.churn_rate * 100).round(1))))

work["AgeGroup"] = pd.cut(work["Age"], bins=[17, 30, 40, 50, 60, 100],
                          labels=["18-30", "31-40", "41-50", "51-60", "60+"])
print("\n--- Churn rate by age group ---")
print((work.groupby("AgeGroup", observed=True)["Exited"].mean() * 100).round(1))
work = work.drop(columns="AgeGroup")

---
## 5. Encode, split, and train

`Geography` and `Gender` are one-hot encoded with `drop_first=True`. The split is **stratified** —
with only ~20% churners, a random split could hand you an unrepresentative test set.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

encoded = pd.get_dummies(work, columns=["Geography", "Gender"], drop_first=True)
y = encoded["Exited"]
X = encoded.drop(columns=["Exited"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print(f"Features: {X.shape[1]}  |  Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Churn rate — train {y_train.mean():.2%}, test {y_test.mean():.2%}")
print(X.columns.tolist())

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=12, min_samples_split=8, min_samples_leaf=3,
        max_features="sqrt", n_jobs=-1, random_state=RANDOM_STATE),
}

results, trained = [], {}
for name, m in models.items():
    m.fit(X_train, y_train)
    trained[name] = m
    pred, proba = m.predict(X_test), m.predict_proba(X_test)[:, 1]
    results.append({"model": name,
                    "accuracy": accuracy_score(y_test, pred),
                    "precision": precision_score(y_test, pred),
                    "recall": recall_score(y_test, pred),
                    "f1": f1_score(y_test, pred),
                    "roc_auc": roc_auc_score(y_test, proba)})

comparison = pd.DataFrame(results).set_index("model").round(4)
display(comparison)
print("\n📋 Copy this table straight into your README's Results section.")

---
## 6. Evaluate the Random Forest

Accuracy alone is misleading here — predicting "nobody churns" already scores ~80% on this dataset.
Watch **recall on the churn class** and **ROC-AUC** instead.

In [ ]:
best = trained["Random Forest"]
pred  = best.predict(X_test)
proba = best.predict_proba(X_test)[:, 1]
auc   = roc_auc_score(y_test, proba)

cv = cross_val_score(best, X, y, cv=5, scoring="accuracy", n_jobs=-1)
print(f"5-fold CV accuracy: {cv.mean():.4f} (+/- {cv.std():.4f})\n")
print(classification_report(y_test, pred, target_names=["Stayed", "Churned"]))

baseline = 1 - y_test.mean()
print(f"Majority-class baseline accuracy: {baseline:.2%}")
print(f"Random Forest accuracy:           {accuracy_score(y_test, pred):.2%}")

In [ ]:
# --- Confusion matrix ---
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Stayed", "Churned"], yticklabels=["Stayed", "Churned"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Random Forest — confusion matrix")
save(fig, "06_confusion_matrix"); plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"Caught {tp} of {tp+fn} real churners. Missed {fn}. False alarms: {fp}.")

In [ ]:
# --- ROC curve ---
fpr, tpr, _ = roc_curve(y_test, proba)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, lw=2.5, color="#D1495B", label=f"Random Forest (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Random guess")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curve"); ax.legend(loc="lower right")
save(fig, "07_roc_curve"); plt.show()

In [ ]:
# --- Feature importance ---
imp = (pd.Series(best.feature_importances_, index=X.columns)
         .sort_values(ascending=False).head(12).reset_index())
imp.columns = ["feature", "importance"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=imp, y="feature", x="importance", hue="feature",
            palette="viridis", legend=False, ax=ax)
ax.set_title("Top drivers of customer churn"); ax.set_ylabel("")
save(fig, "08_feature_importance"); plt.show()

display(imp.head(8))

### Optional: tune the decision threshold

The default 0.5 cutoff maximises accuracy, not business value. If a retention offer costs €50 and a
lost customer costs €500, you want to catch more churners even at the price of some false alarms.

In [ ]:
rows = []
for t in np.arange(0.2, 0.75, 0.05):
    p = (proba >= t).astype(int)
    rows.append({"threshold": round(t, 2),
                 "accuracy": accuracy_score(y_test, p),
                 "precision": precision_score(y_test, p, zero_division=0),
                 "recall": recall_score(y_test, p),
                 "f1": f1_score(y_test, p)})
display(pd.DataFrame(rows).round(3))
print("Lower threshold = catch more churners, more false alarms. Pick one and justify it.")

---
## 7. Export everything for the repo

This writes the model, metrics, charts, the Power BI dataset and a version-matched
`requirements.txt`, then downloads them as one zip.

In [ ]:
# Model bundle — the Streamlit app expects exactly this structure
joblib.dump({"model": best, "columns": X.columns.tolist()}, "models/churn_rf_model.joblib")

metrics = {
    "comparison": results,
    "best_model": "Random Forest",
    "cv_accuracy_mean": float(cv.mean()),
    "cv_accuracy_std": float(cv.std()),
    "top_features": imp.to_dict("records"),
    "n_records": int(len(X)),
}
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)

print("Saved models/churn_rf_model.joblib and models/metrics.json")

In [ ]:
# Scored dataset for Power BI
export = raw.copy()
export["ChurnProbability"] = best.predict_proba(X)[:, 1].round(4)
export["PredictedChurn"]   = (export["ChurnProbability"] >= 0.5).astype(int)
export["RiskBand"] = pd.cut(export["ChurnProbability"], bins=[-0.01, 0.3, 0.6, 1.0],
                            labels=["Low", "Medium", "High"])
export["AgeGroup"] = pd.cut(export["Age"], bins=[17, 30, 40, 50, 60, 100],
                            labels=["18-30", "31-40", "41-50", "51-60", "60+"])
export.to_csv("powerbi/churn_predictions.csv", index=False)

print(export["RiskBand"].value_counts().sort_index())
print("\nSaved powerbi/churn_predictions.csv")

In [ ]:
# requirements.txt pinned to THIS runtime — critical so Streamlit Cloud can load the model
reqs = f"""pandas=={pd.__version__}
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
seaborn=={sns.__version__}
matplotlib=={matplotlib.__version__}
joblib=={joblib.__version__}
streamlit==1.37.0
"""
with open("requirements.txt", "w") as f:
    f.write(reqs)
print(reqs)
print("⚠️  Replace requirements.txt in your repo with this one.")

In [ ]:
# Markdown results table, ready to paste into the README
print("| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |")
print("|---|---|---|---|---|---|")
for r in results:
    star = " ⭐" if r["model"] == "Random Forest" else ""
    print(f"| {r['model']}{star} | {r['accuracy']:.3f} | {r['precision']:.3f} | "
          f"{r['recall']:.3f} | {r['f1']:.3f} | {r['roc_auc']:.3f} |")

In [ ]:
# Zip and download
!zip -rq churn_artifacts.zip models reports powerbi requirements.txt
from google.colab import files
files.download("churn_artifacts.zip")
print("Unzip this into your repo folder, overwriting models/, reports/, powerbi/ and requirements.txt")

---
## ✅ What to do next

1. Unzip `churn_artifacts.zip` into your local repo, overwriting the empty folders.
2. Paste the Markdown results table into the README's **Results** section.
3. Fill in the **Key Findings** bullets with the real numbers from step 4.
4. Run `streamlit run app.py` locally to confirm the model loads, and screenshot it.
5. Build the Power BI dashboard from `powerbi/churn_predictions.csv`.
6. Commit — **force-add the artifacts**, since `.gitignore` excludes data files:
   ```bash
   git add -f models/ reports/figures/ powerbi/churn_predictions.csv
   git add . && git commit -m "Add trained model, figures and dashboard data"
   git push
   ```
7. Deploy on [share.streamlit.io](https://share.streamlit.io) and paste the URL into the README.

**Save this notebook to the repo too** — File → Save a copy in GitHub. A visible notebook is often the
first thing a reviewer opens.
